# Study 956 — The Custody Fee — the teardown

The trend-in-levels estimator with break segmentation, its price-ratio placebo, the currency-free income decomposition, HAC versus block bootstrap, the sign test and leave-one-out, the era cut, the withholding sweep that demonstrates non-identification, and the live synthetic control. Every real number is frozen from `docs/results.md` (fingerprint `ccd87508f03e`).

In [1]:
R = {'start': '2000-01-03', 'end': '2026-06-30', 'n_rows': 6649, 'fp': 'ccd87508f03e', 'n_pairs': 15, 'n_kept': 10, 'n_dropped': 5, 'uk_home_yield_lo': 0.04, 'uk_home_yield_hi': 0.05, 'uk_adr_yield_lo': 3.58, 'uk_adr_yield_hi': 5.82, 'uk_ratio_lo': 97, 'uk_ratio_hi': 114, 'uk_fake_fee': -5.4, 'gap_mean': 13.8, 'gap_median': 7.53, 'gap_sd': 18.48, 'gap_t': 2.36, 'gap_pos': 9, 'gap_n': 10, 'gap_ci_lo': 4.96, 'gap_ci_hi': 25.65, 'gap_frac_le0': 0.0, 'sign_p': 0.0107, 'cents_mean': 9.01, 'cents_median': 5.34, 'cents_ci_lo': 3.09, 'cents_ci_hi': 17.57, 'total_mean': 16.71, 'total_t': 1.92, 'placebo_mean': 3.0, 'placebo_t': 0.58, 'loo_lo': 8.75, 'loo_hi': 15.43, 'loo_t_lo': 1.92, 'loo_t_hi': 2.67, 'loo_worst': 'E', 'loo_best_drop': 'NVS', 'boot_clear': 3, 'boot_names': 'NVS, TSM, TM', 'era_e_gap': 19.99, 'era_e_t': 2.65, 'era_e_pos': 10, 'era_l_gap': 10.23, 'era_l_t': 2.09, 'era_l_pos': 6, 'wht0': 13.8, 'wht0_t': 2.36, 'wht05': -14.67, 'wht1': -43.15, 'wht1_t': -5.5, 'wht15': -71.62, 'wht_cost_lo': 26, 'wht_cost_hi': 96, 'block_t': 1.84, 'block_n': 5, 'drop_nvs_mean': 8.75, 'raw_sw_bp': 944.7, 'raw_sw_t': 1.05, 'clean_rows': 24, 'clean_total': 83728, 'brk006': 20.11, 'brk006_t': 2.56, 'brk010': 16.71, 'brk010_t': 1.92, 'sw_n': 4071, 'sw_adr_sharpe': 0.547, 'sw_loc_sharpe': 0.608, 'sw_gross_bp': 14.6, 'sw_gross_t': 0.11, 'sw_fx30_bp': 12.8, 'sw_mid_bp': -2.2, 'sw_full_bp': -18.5, 'syn_planted': 77.5, 'syn_recovered': 76.31, 'syn_custody': 24.33, 'syn_null': 0.11, 'syn_null8_mean': -0.28, 'syn_null8_sd': 1.69, 'syn_break': 74.51, 'fee_schedule_lo': 1, 'fee_schedule_hi': 5, 'names': (('TTE', 5.1, 1.58), ('SNY', -0.8, -0.42), ('SAP', 10.0, 1.78), ('PHG', 1.9, 2.65), ('ING', 4.4, 2.81), ('E', 31.6, 6.83), ('NVS', 59.3, 18.26), ('NVO', 1.5, 12.1), ('TM', 11.9, 7.69), ('TSM', 13.2, 13.62))}

## The estimator

For each pair, `x_t = log(ADR_TR_t) − log(home_TR_t × FX_t)`. Regress `x` on time in years, **centred within each break segment**, with one dummy per segment; the drag is minus the common slope. Newey-West at **252 lags** because the regression error is the arbitrage band, which is near-unit-root.

Two companion fits: the same regression on the **price-only** ratio (a placebo that must be flat — an ADS is a fixed number of shares), and the **income gap**, `[log(TR) − log(price)]_ADR − [log(TR) − log(price)]_home`, which is the headline because FX multiplies a leg's two closes identically and therefore cancels.

> 💡 *In plain words:* we are not asking whether the receipt's price drifts from the share's. We are asking whether the receipt pays you less.

## The coverage screen and what it removes

In [2]:
print(f"pairs loaded {R['n_pairs']}, kept {R['n_kept']}, dropped {R['n_dropped']} (all LSE)")
print(f"dropped names' HOME yield {R['uk_home_yield_lo']:.2f}-{R['uk_home_yield_hi']:.2f} %/yr "
      f"vs ADR yield {R['uk_adr_yield_lo']:.2f}-{R['uk_adr_yield_hi']:.2f} %/yr "
      f"-> ratio {R['uk_ratio_lo']}-{R['uk_ratio_hi']}x")
print(f"naive drag on those names: {R['uk_fake_fee']:.1f} %/yr -- a vendor artefact, not a fee")
print('the gate reads the HOME leg only, so it cannot select on the estimand')

pairs loaded 15, kept 10, dropped 5 (all LSE)
dropped names' HOME yield 0.04-0.05 %/yr vs ADR yield 3.58-5.82 %/yr -> ratio 97-114x
naive drag on those names: -5.4 %/yr -- a vendor artefact, not a fee
the gate reads the HOME leg only, so it cannot select on the estimand


## Headline — pooled across names

Names are the observations; the estimation error is dominated by each pair's own arbitrage band while the *fee* is a common institutional parameter, so the cross-name statistic tests "is the average leak non-zero", not any one name.

In [3]:
print(f"total drag    mean {R['total_mean']:+6.2f} bp/yr  t {R['total_t']:+5.2f}")
print(f"price placebo mean {R['placebo_mean']:+6.2f} bp/yr  t {R['placebo_t']:+5.2f}  <- flat, as it must be")
print(f"income gap    mean {R['gap_mean']:+6.2f} bp/yr  median {R['gap_median']:+6.2f}  "
      f"sd {R['gap_sd']:.2f}  t {R['gap_t']:+5.2f}  positive {R['gap_pos']}/{R['gap_n']}")
print(f"  name-bootstrap 95% CI [{R['gap_ci_lo']:+.2f}, {R['gap_ci_hi']:+.2f}] bp/yr, "
      f"share<=0 {R['gap_frac_le0']:.3f}")
print(f"  sign test p = {R['sign_p']:.4f}")
print(f"  in cents/ADS/yr: mean {R['cents_mean']:.2f}, median {R['cents_median']:.2f}, "
      f"CI [{R['cents_ci_lo']:.2f}, {R['cents_ci_hi']:.2f}] vs a published "
      f"{R['fee_schedule_lo']}-{R['fee_schedule_hi']} c/ADS/yr schedule")

total drag    mean +16.71 bp/yr  t +1.92
price placebo mean  +3.00 bp/yr  t +0.58  <- flat, as it must be
income gap    mean +13.80 bp/yr  median  +7.53  sd 18.48  t +2.36  positive 9/10
  name-bootstrap 95% CI [+4.96, +25.65] bp/yr, share<=0 0.000
  sign test p = 0.0107
  in cents/ADS/yr: mean 9.01, median 5.34, CI [3.09, 17.57] vs a published 1-5 c/ADS/yr schedule


## Where it breaks down

The HAC *t* is generous and the block bootstrap is not. Report both. And the cross-name *t* assumes ten independent issuers, which is false: six of the ten are euro-area names sharing a currency, a treaty rate and a dividend calendar, so the honest unit of resampling is the currency block, not the name.

In [4]:
print(f"leave-one-out mean range [{R['loo_lo']:.2f}, {R['loo_hi']:.2f}] bp/yr, "
      f"t range [{R['loo_t_lo']:+.2f}, {R['loo_t_hi']:+.2f}]")
print(f"  dropping {R['loo_worst']} alone takes t under 2; dropping {R['loo_best_drop']} "
      f"(two spin-offs) halves the mean to {R['drop_nvs_mean']:.2f} bp/yr")
print(f"collapse the 6 euro names to one obs (EUR/CHF/DKK/JPY/TWD): "
      f"t {R['block_t']:+.2f} on n = {R['block_n']}  <- below the bar")
print(f"per-name 63-day block bootstrap clears zero on {R['boot_clear']}/{R['gap_n']} "
      f"names ({R['boot_names']}) -- name by name the leak is invisible")
print(f"price placebo contaminated on PHG (+32.5), NVS (+23.2), TSM (-30.6) bp/yr "
      f"-> the TOTAL-drag column is unusable for those; the income gap is not")
print('per-name HAC t on the income leg is NOT to be trusted (near-deterministic')
print('  staircase => tiny residual => NVO reads t +12.10 on a 1.5 bp/yr gap)')
print('survivorship: every issuer is still dual-listed in 2026 -- a survivor panel')

leave-one-out mean range [8.75, 15.43] bp/yr, t range [+1.92, +2.67]
  dropping E alone takes t under 2; dropping NVS (two spin-offs) halves the mean to 8.75 bp/yr
collapse the 6 euro names to one obs (EUR/CHF/DKK/JPY/TWD): t +1.84 on n = 5  <- below the bar
per-name 63-day block bootstrap clears zero on 3/10 names (NVS, TSM, TM) -- name by name the leak is invisible
price placebo contaminated on PHG (+32.5), NVS (+23.2), TSM (-30.6) bp/yr -> the TOTAL-drag column is unusable for those; the income gap is not
per-name HAC t on the income leg is NOT to be trusted (near-deterministic
  staircase => tiny residual => NVO reads t +12.10 on a 1.5 bp/yr gap)
survivorship: every issuer is still dual-listed in 2026 -- a survivor panel


## Era cut (split 2015-01-01) and break-threshold sweep

In [5]:
print(f"2000-2014: income gap {R['era_e_gap']:+6.2f} bp/yr (t {R['era_e_t']:+5.2f}), "
      f"positive {R['era_e_pos']}/{R['gap_n']}")
print(f"2015-2026: income gap {R['era_l_gap']:+6.2f} bp/yr (t {R['era_l_t']:+5.2f}), "
      f"positive {R['era_l_pos']}/{R['gap_n']}  <- same sign, half the size")
print()
print(f"break threshold 0.06 -> total drag {R['brk006']:+6.2f} (t {R['brk006_t']:+5.2f})")
print(f"break threshold 0.10 -> total drag {R['brk010']:+6.2f} (t {R['brk010_t']:+5.2f})  "
      f"(0.15 and 0.25 identical: exactly ONE level shift exists in the kept "
      f"panel -- ING, 2009-11-23, the state-aid rights issue)")

2000-2014: income gap +19.99 bp/yr (t +2.65), positive 10/10
2015-2026: income gap +10.23 bp/yr (t +2.09), positive 6/10  <- same sign, half the size

break threshold 0.06 -> total drag +20.11 (t +2.56)
break threshold 0.10 -> total drag +16.71 (t +1.92)  (0.15 and 0.25 identical: exactly ONE level shift exists in the kept panel -- ING, 2009-11-23, the state-aid rights issue)


## The ASSUMPTION sweep — the fee/tax split is not identified

The withholding rate is the only non-tape input (treaty rates, 15 % to 21 %). Scaling it moves the residual "custody" term straight through zero and out the other side.

The intended anchor was the **UK** — no dividend withholding tax at all, so a UK pair's income gap *is* the custody fee by law. All five UK pairs are London listings and all five die on the coverage screen. Nothing identifies the split.

> ⚠️ **And the same logic bounds the fee, not just the tax.** If the vendor's per-ADS dividend is the *gross declared* amount — which is what this sweep proves — then a depositary fee billed through DTC as a separate account charge never enters the series either. The measured gap is an **upper bound on the fee**, and is equally consistent with the depositary's FX-conversion spread, per-ADS rounding, and feed differences on special dividends and spin-offs. This study measures a leak; it does not name it.

In [6]:
print(f"0.0 x treaty -> residual custody {R['wht0']:+7.2f} bp/yr (t {R['wht0_t']:+5.2f})")
print(f"0.5 x treaty -> residual custody {R['wht05']:+7.2f} bp/yr")
print(f"1.0 x treaty -> residual custody {R['wht1']:+7.2f} bp/yr (t {R['wht1_t']:+5.2f})")
print(f"1.5 x treaty -> residual custody {R['wht15']:+7.2f} bp/yr")
print()
print(f"treaty withholding alone would cost {R['wht_cost_lo']}-{R['wht_cost_hi']} bp/yr "
      f"on these yields -- 3x to 7x the WHOLE measured gap of {R['gap_mean']:.1f} bp/yr")
print('=> the withholding is absent from the ADR total-return series; only the')
print('   0.0x column is defensible, and it makes the gap an UPPER BOUND on the fee')

0.0 x treaty -> residual custody  +13.80 bp/yr (t +2.36)
0.5 x treaty -> residual custody  -14.67 bp/yr
1.0 x treaty -> residual custody  -43.15 bp/yr (t -5.50)
1.5 x treaty -> residual custody  -71.62 bp/yr

treaty withholding alone would cost 26-96 bp/yr on these yields -- 3x to 7x the WHOLE measured gap of 13.8 bp/yr
=> the withholding is absent from the ADR total-return series; only the
   0.0x column is defensible, and it makes the gap an UPPER BOUND on the fee


## The only traded leg — own the home line instead

Equal-weight baskets, excess-of-cash versus BIL **on both legs**, long-only (no short leg, hence no borrow), one-way FX conversion × NAV executed at **t+1** on a decision formed at *t*. Both frictions are assumptions and both are swept.

> ⚠️ **This leg is a measurement, not a backtest.** Two reasons. (1) It inherits the bad-print screen from `data.load_pair`, whose rolling median is *centred* and therefore peeks forward. That is inert on the headline — the pooled income gap is identical with the filter off — but decisive here: switch it off and the same race reads **+944.7 bp/yr at HAC t = +1.05**, because 24 corrupt rows out of 83,728 (0.03 %) dominate an arithmetic mean of daily returns. (2) Rebalancing turnover is charged on neither leg, and only the home leg would really pay FX on every trade. Both readings are Mirage; the cleaned one is the honest magnitude.

In [7]:
print(f"n = {R['sw_n']} common days (BIL inception gates the cash leg)")
print(f"ADR basket excess-of-cash Sharpe {R['sw_adr_sharpe']:+.3f} "
      f"vs home basket {R['sw_loc_sharpe']:+.3f}")
print(f"gross          : {R['sw_gross_bp']:+6.1f} bp/yr  HAC t {R['sw_gross_t']:+.2f}")
print(f"30 bp FX       : {R['sw_fx30_bp']:+6.1f} bp/yr")
print(f"30 bp + 15 bp/yr custody: {R['sw_mid_bp']:+6.1f} bp/yr  <- already negative")
print(f"50 bp + 30 bp/yr custody: {R['sw_full_bp']:+6.1f} bp/yr")
print()
print(f"LOOK-AHEAD CHECK -- hygiene filter OFF: {R['raw_sw_bp']:+.1f} bp/yr "
      f"HAC t {R['raw_sw_t']:+.2f}  ({R['clean_rows']} of {R['clean_total']:,} "
      f"rows = {100*R['clean_rows']/R['clean_total']:.3f}% drive the difference)")
print('  same check on the HEADLINE income gap: +13.80 bp/yr either way -- inert')

n = 4071 common days (BIL inception gates the cash leg)
ADR basket excess-of-cash Sharpe +0.547 vs home basket +0.608
gross          :  +14.6 bp/yr  HAC t +0.11
30 bp FX       :  +12.8 bp/yr
30 bp + 15 bp/yr custody:   -2.2 bp/yr  <- already negative
50 bp + 30 bp/yr custody:  -18.5 bp/yr

LOOK-AHEAD CHECK -- hygiene filter OFF: +944.7 bp/yr HAC t +1.05  (24 of 83,728 rows = 0.029% drive the difference)
  same check on the HEADLINE income gap: +13.80 bp/yr either way -- inert


## Live synthetic control — recovery, silence on the null, immunity to a ratio break

**Synthetic only.** Planted fee: the estimator must recover it and split it correctly when the withholding rate is known. Null: it must report ~zero. Planted ADS-ratio step: the segment fixed effects must absorb it.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from adr_drag import data, strategy as st

fr, tr = data.synthetic_panel(n_names=8, drag_bps_per_year=25.0, signal_strength=1.0)
w = {k: tr['per_name'][k]['wht'] for k in fr}
d = st.synthetic_detect(fr, w)
print('planted gap %+.2f -> recovered %+.2f bp/yr (t %+.1f); residual custody %+.2f (planted %.1f)'
      % (tr['planted_gap_per_year']*1e4, d['income_gap']['mean']*1e4,
         d['income_gap']['t'], d['custody']['mean']*1e4, tr['custody_drag_per_year']*1e4))
print('price placebo %+.2f bp/yr (the fee is netted from the dividend, never the price)'
      % (d['price_drift']['mean']*1e4))

nulls = []
for s in range(6):
    f0, t0 = data.synthetic_panel(n_names=6, n_years=10, drag_bps_per_year=25.0,
                                  signal_strength=0.0, seed=956 + 11*s)
    nulls.append(st.synthetic_detect(f0, {k: 0.0 for k in f0})['income_gap']['mean'])
nulls = np.array(nulls) * 1e4
print('null x6 seeds: mean %+.2f bp/yr (sd %.2f), |mean| >= 5 bp on %d/6'
      % (nulls.mean(), nulls.std(ddof=1), (np.abs(nulls) >= 5).sum()))

fb, tb = data.synthetic_panel(n_names=4, drag_bps_per_year=25.0, ratio_break=0.70)
db = st.synthetic_detect(fb, {k: tb['per_name'][k]['wht'] for k in fb})
print('with a planted 0.70-log ADS-ratio step: total drag %+.2f bp/yr (planted %.1f)'
      % (db['drag']['mean']*1e4, tb['planted_gap_per_year']*1e4))

planted gap +76.31 -> recovered +76.44 bp/yr (t +89.6); residual custody +24.45 (planted 25.0)
price placebo +1.06 bp/yr (the fee is netted from the dividend, never the price)


null x6 seeds: mean -0.37 bp/yr (sd 1.06), |mean| >= 5 bp on 0/6


with a planted 0.70-log ADS-ratio step: total drag +75.58 bp/yr (planted 76.3)


## Verdict

- **Signal — Weak.** The income shortfall is real and correctly signed: **+13.80 bp/yr** pooled (median +7.53), 9/10 positive (sign test *p* = 0.011), name-bootstrap CI **[+4.96, +25.65] bp/yr** with share ≤ 0 of 0.000, positive in both eras (+20.0 / +10.2), and a median **5.3 c/ADS/yr** in the range of the published 1-5 c schedules. It is not robust: *t* = +2.36 → +1.92 on leave-one-out and +1.84 on 5 currency blocks, 3/10 names clear a block-bootstrap CI, dropping Novartis halves the mean to +8.75, the price placebo is contaminated on three names, 5/15 pairs are unusable on a vendor defect, and the panel is a **survivor panel**. Nor is the leak attributable: assumed withholding 26-96 bp/yr exceeds the whole gap, so the tax is absent from the tape — and by the same argument a DTC-billed depositary fee would be too, making +13.80 bp/yr an **upper bound** rather than a fee. The synthetic control recovers a planted 77.5 bp/yr as 76.31, is silent on the null (mean -0.28 bp/yr over 8 seeds) and survives a planted ratio break (74.51) — the ambiguity is the tape's, not the harness's.
- **Tradability — Mirage.** The home basket beats the ADR basket by **+14.6 bp/yr gross at HAC *t* = +0.11** and turns negative (-2.2 bp/yr) at a 15 bp/yr foreign safekeeping charge, with rebalancing turnover charged on neither leg and a forward-peeking hygiene filter baked in (without it: +944.7 bp/yr at *t* = +1.05, driven by 0.029 % of the rows). A real, small, unavoidable cost of the wrapper — not an edge.